In [1]:
# Import Libraries
import pandas as pd
from bs4 import BeautifulSoup
import requests
import smtplib
from datetime import time, datetime, timedelta
import random, re

# Global Variables
global bullet_characters 
global salary_descriptors 
bullet_characters = ['-', '•', '◦', '▪', '▸', '➤', '★', '*']
salary_descriptors = ['$','pay','paid','salary','compensation']

In [2]:
# Extract HTML - Paginated Func
def extract(page):    
    url = f'https://www.teamworkonline.com/jobs-in-sports?employment_opportunity_search%5Bsort_by%5D=most_recent&page={page}'
    user_agents_list = [
    'Mozilla/5.0 (iPad; CPU OS 12_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.83 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.51 Safari/537.36',
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36"]
    
    headers = {'User-Agent': random.choice(user_agents_list)}

    page = requests.get(url,headers = headers)

    soup = BeautifulSoup(page.content,'html.parser')
    return(soup)

In [3]:
# Extract HTML - Inner Link (a tag) Func
def extract_inner(url):    
    user_agents_list = [
    'Mozilla/5.0 (iPad; CPU OS 12_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/15E148',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.83 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.51 Safari/537.36',
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36"]
    
    headers = {'User-Agent': random.choice(user_agents_list)}

    page = requests.get(url,headers = headers)

    soup = BeautifulSoup(page.content,'html.parser')
    return(soup)

In [195]:
def transform(jobs_soup):
    all_jobs = jobs_soup.find_all(class_ = "browse-jobs-card")
    raw_jobs = []
    for job in all_jobs:
##################################################################### Scraping Basic info from Job cards ############################################################################
        try:
            raw_job={
                'logo_image': job.find('img').get('src') if job.find('img') else None,
                'job_url': 'https://www.teamworkonline.com' + job.find('a').get('href') if job.find('a') else None,
                'title': job.find(class_ = "margin-none").get_text().strip() if job.find(class_ = "margin-none") else None,
                'company': job.find(class_ = "browse-jobs-card__content--organization").get_text().strip() if job.find(class_ = "browse-jobs-card__content--organization") else None,
                'location_info': job.find(class_ = "trending__content--small").get_text().strip() if job.find(class_ = "trending__content--small") else None,
                'experience_level': job.find(class_ = "browse-jobs-card__content--small").get_text().strip() if job.find(class_ = "browse-jobs-card__content--small") else None  
            }
####################################################################### Extracting info from the individual job site #################################################################     
            job_soup = extract_inner(raw_job['job_url'])
            raw_job['salary_info'] = salary_fetcher(job_soup)
            raw_job['details'] = detail_fetcher(job_soup)
                       
################################################### Scraping time based info from Job cards and doing minor extractions  #############################################################    
            posting_number_list = job.find_all(class_ = "browse-jobs-card__scoreboard")
            if len(posting_number_list)>= 2:
                raw_job['posting_numbers'] = posting_number_list[0].get_text().strip() + posting_number_list[1].get_text().strip()
            else:
                raw_job['posting_numbers'] = None
            raw_job['posting_date_identifier'] = job.find(class_ = "trending__content--small trending__scoreboard--time margin-left--xx-small").get_text().strip().split(' ')[0]
            raw_job['scrape_datetime'] = datetime.now()
            
            jobs_list.append(raw_job)
            
        except Exception as e:
            print(f"Error during job scrape: {e}")
            continue
    return

In [193]:
job_posting = {}
job_soup = extract_inner('https://www.teamworkonline.com/hockey-jobs/hockeyjobs/nhl-league-office/royalty-analyst-2163333')
job_posting['salary_info'] = salary_fetcher(job_soup)
job_posting['details'] = detail_fetcher(job_soup)

Need to Standardize Divs
<p>The royalty analyst will be responsible for the day-to-day processing of consumer products royalty statements. Tasks include:</p>



<p>Required</p>



<p>Preferred</p>



<p>Education/Certifications</p>



<p>Required Skills</p>



<p>These core competencies reflect the underlying values that are necessary to represent the National Hockey League:</p>





In [194]:
job_posting

{'salary_info': '$60,000 - $63,000 / year',
 'details':   The royalty analyst will be responsible for the day-to-day processing of consumer products royalty statements. Tasks include:  \
 0        Processing statements in the royalty system                                                                              
 1  Reviewing royalties reported for assigned lice...                                                                              
 2  Analyzing revenue generated from royalty state...                                                                              
 3  Reviewing and applying cash received from lice...                                                                              
 4  Communicate with licensees regarding their roy...                                                                              
 5  Assisting with the preparation of royalty repo...                                                                              
 6  Assist with testi

In [199]:
# This is code to find salary in the top portion of the sit
def salary_fetcher(job_soup):
    salary_desc = None
    try:
        salary_flag = job_soup.find_all('div',class_ = 'margin-right--x-small')
        for i in range(0,len(salary_flag)):
            if any(descriptor in str(salary_flag[i]) for descriptor in salary_descriptors):
                salary_desc = salary_flag[i].get_text()
                #print(salary_flag[i].get_text())
            else:
                pass
    except Exception as e:
        print(f"Error during job scrape: {e}")
    return(salary_desc)

In [179]:
####################################### THIS IS THE NEW APPROACH AS OF 3/17 ##############################################
#text = "This is a sample test to see what will happen to be honest: \n\n\n- This is bullet 1.\n\n\n\n- This is bullet 2.\n- This is bullet 3.\n- This is bullet 4.\n- This is bullet 5. \n This is extra text Now that Idk what will happen to."

def wrap_bullets(html_text):
    if(html_text.find('ul')):
        for ul in html_text.find_all('ul'):
            for li in ul:
                li.string = '- ' + li.get_text()
    else:
        pass
    
    text = html_text.get_text(separator = '\n')
    
    lines = text.split('\n')
    result = []
    in_list = False
    
    for line in lines:
        stripped_line = line.strip()
        
        if in_list and stripped_line == '':
            continue
    
        if stripped_line and stripped_line[0] in bullet_characters:
            if not in_list:
                preceding_line = next((line for line in reversed(result) if line.strip()), None)
                result.append(f'<p>{preceding_line}</p>')
                result.append('<ul>')
                in_list = True
            content = stripped_line[1:].strip()
            result.append(f'<li>{content}</li>')
        else:
            if in_list:
                result.append('</ul>')
                in_list = False
            result.append(stripped_line)
            
    if in_list:
        result.append('</ul>')
    
    final_result = '\n'.join(result)
    
    soup = BeautifulSoup(final_result,'html.parser')

    return(soup.find_all('p'))

In [203]:
def detail_fetcher(job_soup):
    try:
        job_inner = job_soup.find('div', class_='opportunity-preview__body')
        job_details = job_inner.find_all('p')
        
        if job_details:
            if job_inner.find('ul'):
                print('Standard HTML')
                p_tag = True
            else:
                p_tag = False
                print('Need to standardize Ps')
        else:
            p_tag = False
            print('Need to Standardize Divs')
        
        if (p_tag):
            pass
        else:
            job_details = wrap_bullets(job_inner)
        
        data = {}
        headers = []
        for section in job_details:
            
            if not section:
                continue
            
            details = []
                
            header_text = section.get_text()
            next_tag = section.find_next_sibling()
            
            if (next_tag and next_tag.name in ['ul', 'ol']):
                headers.append(header_text)
                items = next_tag.find_all('li')
                details = [item.get_text(strip=True) for item in items]
            
                data[header_text] = details 
                    
        
        # Convert to single-row DataFrame
        details_df = pd.DataFrame({k: pd.Series(v) for k, v in data.items()})
    except:
        details_df = []
    return(details_df)

In [181]:
headers

['The royalty analyst will be responsible for the day-to-day processing of consumer products royalty statements. Tasks include:',
 'Required',
 'Preferred',
 'Education/Certifications',
 'Required Skills',
 'These core competencies reflect the underlying values that are necessary to represent the National Hockey League:']

In [182]:
# CODE TO PULL OFF RELEVANT INFO FROM DETAIL SCRAPE
df.filter(regex = '(?i)requ|respon|qual|look|skill|know|job|function|Elig|edu|exper|duties|duty',axis=1)

,The royalty analyst will be responsible for the day-to-day processing of consumer products royalty statements. Tasks include:,Required,Education/Certifications,Required Skills
0,Processing statements in the royalty system,Advanced Microsoft Excel skills,"Minimum of Bachelor’s degree, Accounting or Fi...",Excellent verbal and written skills
1,Reviewing royalties reported for assigned lice...,NaN,NaN,Strong analytical and organizational skills an...
2,Analyzing revenue generated from royalty state...,NaN,NaN,Able to handle multiple tasks simultaneously
3,Reviewing and applying cash received from lice...,NaN,NaN,Take initiative and develop creative solutions...
4,Communicate with licensees regarding their roy...,NaN,NaN,NaN
5,Assisting with the preparation of royalty repo...,NaN,NaN,NaN
6,Assist with testing new releases of the royalt...,NaN,NaN,NaN
7,Ad hoc special projects,NaN,NaN,NaN


In [183]:
details_df = []
for header in headers:
    details_df.append({header: df[header].str.split('\n').explode(header)})
details_df = pd.DataFrame(details_df)
details_df

,The royalty analyst will be responsible for the day-to-day processing of consumer products royalty statements. Tasks include:,Required,Preferred,Education/Certifications,Required Skills,These core competencies reflect the underlying values that are necessary to represent the National Hockey League:
0,0 Processing statements in the royalt...,NaN,NaN,NaN,NaN,NaN
1,NaN,0 Advanced Microsoft Excel skills 1 ...,NaN,NaN,NaN,NaN
2,NaN,NaN,0 Experience with consumer products licensi...,NaN,NaN,NaN
3,NaN,NaN,NaN,"0 Minimum of Bachelor’s degree, Accounting ...",NaN,NaN
4,NaN,NaN,NaN,NaN,0 Excellent verbal and writte...,NaN
5,NaN,NaN,NaN,NaN,NaN,0 Accountability 1 ...


In [200]:
# Will use this once deployed pages = (1,11)
pages = ((1,3))
jobs_list = []
for page in range(pages[0],pages[1]):
    recent_jobs_html = extract(page)
    transform(recent_jobs_html)

Need to Standardize Divs
<p>Figure Skating Business Operations &amp; Innovation</p>



<p>advancements, skating industry partnerships, standardizing processes)</p>



<p>Impact 2030</p>



<p>successful candidate.</p>



Standard HTML
<p><strong>Company Information</strong><br/>For more than 20 years, AEG has played a pivotal role in transforming sports and live entertainment. Annually, we host more than 160 million guests, promote more than 10,000 shows and present more than 22,000 events around the world. We are committed to innovation, artistry, and community, and leverage the power of our 300+ venues, leading sports franchises, marquee music brands, integrated entertainment districts, premier ticketing platform and global sponsorship activations, to create memorable moments that give the world reason to cheer.<br/><br/>Our business is interwoven with the human mind and heart, and we strive to build a diverse and inclusive company that reflects the artists, athletes, and fans that w

Standard HTML
<p>Job Title: Sales Manager</p>



<p>Department: Sales &amp; Services</p>



<p>Reports To: Assistant Director of Sales</p>



<p>Supervises: None</p>



<p>FLSA Status: Salaried, Exempt</p>



<p></p>



<p>Why the Colorado Convention Center is a great place to work:</p>



<p>$500 Signing Bonus after 30 days of service and an additional $500 bonus upon successful completion of 90 days of service!</p>



<p>Generous Paid Time Off and Holiday Pay</p>



<p>Health, dental, vision insurance, eligible upon hire</p>



<p>401(k) investment plan, with an employer match of up to 4%</p>



<p>Healthcare reimbursement and flexible spending plans</p>



<p>Employer-paid and supplemental life insurance</p>



<p>Short- and long-term disability insurance available</p>



<p>RTD Eco Pass: As a Full-time employee the Colorado Convention Center currently offers an RTD Eco Pass.</p>



<p>Tuition reimbursement program</p>



<p>Employee assistance program</p>



<p></p>



<p>Compensat

Need to Standardize Divs
<p>RESPONSIBILITIES:</p>



<p>CHARACTERISTICS/QUALIFICATIONS:</p>



Need to Standardize Divs
<p>Responsibilities:</p>



<p>Requirements</p>



<p>We’re excited to offer full-time positions with fantastic benefits that support your well-being and professional growth. Here’s what you can look forward to:</p>



Standard HTML
<p>Monumental Sports &amp; Entertainment (MSE) is one of the largest integrated sports and entertainment companies in the country with one of the most diverse partnership groups in all of sports. MSE owns and operates seven major and minor sports teams: 2018 NHL Stanley Cup Champion Washington Capitals, NBA’s Washington Wizards, WNBA’s 2019 Championship Washington Mystics, NBA G League’s Capital City Go-Go, 2021 &amp; 2020 NBA 2K League Champion Wizards District Gaming and Caps Gaming, an esports sub-brand of the Capitals. Additionally, it holds a significant investment in a seventh professional team, Team Liquid, an endemic esports team o

Standard HTML
<p>Summary:</p>



<p>Under general supervision of the Parking Supervisor, responsible for overall supervision of parking operations during events encompassing efficient daily operations and activities in all Legends Global parking facilities.  Administers the operational facet of the division by ensuring staff preparedness for opening and closing the garages, collecting revenue, resolving problems, receiving and disseminating information and instructions, preparing reports, overseeing money, ensuring internal and public safety by performing the following duties personally and through subordinate personnel.</p>



<p></p>



<p>ESSENTIAL DUTIES AND RESPONSIBILITIES</p>



<p>Include the following. Other duties and responsibilities may be assigned.</p>



<p></p>



<p></p>



<p>Supervisory Responsibilities </p>



<p>Directly supervises employees in the Parking Department including full-time and part-time cashiers and parking attendants. Carries out supervisory responsib

Standard HTML
<p>It's fun to work in a company where people truly BELIEVE in what they're doing!</p>



<p></p>



<p>We're committed to bringing passion and customer focus to the business.</p>



<p></p>



<p>JOB SUMMARY:</p>



<p>The Texas Rangers are looking for a Manager of Venue Operations to join our team! The Manager of Venue Operations is responsible for overseeing the day-to-day operations of Choctaw Stadium, Ballpark Circle, and on property tenant spaces. Their primary focus is to ensure that both proactive and reactive maintenance efforts are carried out effectively in the stadium and tenant spaces. This involves monitoring mechanical, electrical, and plumbing systems, as well as lighting, elevators, and general maintenance throughout the mentioned properties. The Manager utilizes a computerized maintenance management system (CMMS) to diligently document work orders and repairs. A key aspect of this role is collaborating closely with building tenants, maintenance, facility

...

Standard HTML
<p>The Detroit Lions currently have an opening for a <strong>Production Intern</strong>. Reporting to the Production Manager, the Production Intern will be responsible for supporting media content for all Detroit Lions content platforms. This will include videography, editing and overall production of engaging video content for the team’s digital properties as well as facilitation of ongoing media availability sessions.</p>



<p>The Production Intern is a season-long role supporting the 2026 football season, with an anticipated end in January 2027, with potential for post-season extension. Candidates will need to be able to work 5 days per week plus some weekends. Due to the unique nature of this position, the candidate must be able to work extended hours during the NFL season and training camp, which will include working on nights, weekends, and holidays.</p>



<p>This position is based in the Meijer Performance center in Allen Park, MI, but will also require travel to

Need to Standardize Divs
<p>Keynote Speaker Session:</p>



<p>Organizations Scheduled to Attend So Far:</p>



Need to Standardize Divs
<p>Required:</p>



<p>Preferred:</p>



<p>Core Competencies</p>



Need to Standardize Divs
<p>Roles and Responsibilities</p>



<p>Do you want to be a superstar in the sports business? That’s the first qualification. If you answered yes, then answer two more questions: 1) do you have a strong work ethic, and 2) do you have a desire to learn and excel?</p>



<p>Benefits &amp; Compensation</p>



Need to Standardize Divs
<p>Roles and Responsibilities</p>



<p>Do you want to be a superstar in the sports business? That’s the first qualification. If you answered yes, then answer two more questions: 1) do you have a strong work ethic, and 2) do you have a desire to learn and excel?</p>



<p>Benefits &amp; Compensation</p>



Need to Standardize Divs
<p>Sponsorship Sales &amp; Partnerships:</p>



<p>Ticket Sales &amp; Fan Engagement:</p>



<p>Additio

Need to Standardize Divs
<p>The West Henderson Fieldhouse will feature:</p>



<p>Qualifications:</p>



Standard HTML
<p><strong>About IMG Academy</strong><br/>Named one of the Best and Brightest Companies to Work For in the Nation in 2024, IMG Academy is the world's leading sports education brand, providing a holistic education model that empowers student-athletes to win their future, preparing them for college and for life. IMG Academy provides growth opportunities for all student-athletes through an innovative suite of on-campus and online experiences:  </p>



<p><strong>Position Summary: </strong>The <strong>Tennis Operations Coordinator </strong>is the key operational support leader ensuring smooth execution of tennis program logistics, athlete experience, event coordination, and administrative systems.</p>



<p><strong>Key Responsibilities:</strong></p>



<p><strong>Knowledge, Skills and Abilities:</strong>   </p>



<p><strong>Preferred Skills</strong>:</p>



<p> <strong>Ph

Standard HTML
<p></p>



<p>Join the team at Hershey Lodge, an award-winning resort best known for being warm, welcoming, and distinctly Hershey. Offering 665 guest rooms and 100,000 square feet of function space, Hershey Lodge provides convenience and comfort for families and guests of all ages. We hope you'll enjoy the sweet hospitality and iconic chocolate details around every corner.</p>



<p>This position is responsible for all aspects of food service and guest satisfaction with the Bears' Den restaurant located at the Hershey Lodge. </p>



<p>As a Part-Time Team Member, you will enjoy sweet perks like FREE admission and parking to Hersheypark, discounts on food &amp; shopping, and more as soon as you receive your Employee ID!</p>



<p><strong>Job Duties (Duties marked with an asterisk are essential functions of this job):</strong></p>



<p>Greeting all guests in a friendly, efficient, and timely manner </p>



<p>Having a thorough knowledge of all menu and related items to in

Error during job scrape: 'NoneType' object has no attribute 'find_all'
Error during job scrape: 'NoneType' object has no attribute 'find_all'
Error during job scrape: 'NoneType' object has no attribute 'find_all'
Error during job scrape: 'NoneType' object has no attribute 'find_all'
Error during job scrape: 'NoneType' object has no attribute 'find_all'
Error during job scrape: 'NoneType' object has no attribute 'find_all'
Error during job scrape: 'NoneType' object has no attribute 'find_all'
Error during job scrape: 'NoneType' object has no attribute 'find_all'
Error during job scrape: 'NoneType' object has no attribute 'find_all'
Error during job scrape: 'NoneType' object has no attribute 'find_all'
Error during job scrape: 'NoneType' object has no attribute 'find_all'
Error during job scrape: 'NoneType' object has no attribute 'find_all'
Error during job scrape: 'NoneType' object has no attribute 'find_all'
Error during job scrape: 'NoneType' object has no attribute 'find_all'
Error 

In [202]:
temp_df = pd.DataFrame(jobs_list)
temp_df['details']

0           Figure Skating Business Operations & Inn...
1                                      Essential Fun...
2                 Empty DataFrame
Columns: []
Index: []
3                                        Why join ou...
4                      ESSENTIAL DUTES AND RESPONSIB...
5        Why the Colorado Convention Center is a gre...
6                     ESSENTIAL DUTES AND RESPONSIBI...
7                                         RESPONSIBI...
8                                         Responsibi...
9       MSE proudly promotes its core values for all...
10                                    Job Responsibi...
11       Include the following. Other duties may be ...
12      Include the following. Other duties may be a...
13                                                  ...
14                                      ESSENTIAL FU...
15      Duties and responsibilities for the Box Offi...
16                     Essential Duties & Responsibi...
17      We’re looking for passionate, energetic 

In [97]:
team_work_df = pd.DataFrame(jobs_list)

team_work_df['city'] = team_work_df['location_info'].str.split('·').str[0].str.strip()
team_work_df['state'] = team_work_df['location_info'].str.split('·').str[1].str.strip()

In [110]:
team_work_df.loc[team_work_df['state'].str.len()>2,['title','location_info','city']]

,title,location_info,city
91,HR Business Partner,London · United Kingdom · Hybrid,London
98,"Major Gift Officer, West Coast",CA · Remote,CA
131,Event Operations + Merchandising Intern,FL · Hybrid,FL
132,Event Merchandising Manager (On-Site),FL · Hybrid,FL
134,Creative Strategist,United States · Remote,United States
135,Account Executive,United States · Remote,United States
231,Event Operations + Merchandising Intern,FL · Hybrid,FL
232,Event Merchandising Manager (On-Site),FL · Hybrid,FL
234,Creative Strategist,United States · Remote,United States
235,Account Executive,United States · Remote,United States


In [25]:
# Will use this once deployed pages = ((1,3),(3,5),(5,7),(7,9),(9,11))
pages = ((1,3))
for page_range in pages:
    for page_ind in range(page_range[0],page_range[1]):
        recent_jobs_html = extract(page_ind)
        transorm(recent_jobs_html)

1
2
3
4
5
6
7
8
9
10


In [12]:
### This is a copy of scaper code so I can split up some processing

all_jobs = soup.find_all(class_ = "browse-jobs-card")

for job in all_jobs:
##################################################################### Scraping Basic info from Job cards ####################################################
    
    logo_image = job.find('img').get('src')
    job_url = 'https://www.teamworkonline.com' + job.find('a').get('href')
    title = job.find(class_ = "margin-none").get_text().strip()
    company = job.find(class_ = "browse-jobs-card__content--organization").get_text().strip()
    location_info = job.find(class_ = "trending__content--small").get_text().strip()
    # Will need to split as such location_info.split('·')[0])
    experience = job.find(class_ = "browse-jobs-card__content--small").get_text().strip()
    
################################################### Scraping time based info from Job cards and doing minor calcualtions ####################################
    posting_number_list = job.find_all(class_ = "browse-jobs-card__scoreboard")
    posting_numbers = posting_number_list[0].get_text().strip() + posting_number_list[1].get_text().strip()
    posting_date_identifier = job.find(class_ = "trending__content--small trending__scoreboard--time margin-left--xx-small").get_text().strip().split(' ')[0]
    if posting_date_identifier.endswith('s'):
        pass
    else:
        posting_date_identifier = posting_date_identifier + ('s')
    posting_timestamp = datetime.now() - timedelta(**{posting_date_identifier: int(posting_numbers)})

'02/27/2026 12:58:28'

tests


In [20]:
######################### OLD CODE - N0 LONGER USED BUT KEPT BECAUSE ALOT OF WORK ##########################################

#.find_all(['p','strong']) •
data = {}
headers = []
for section in job_details:
    if (p_tag):
        full_text = section
        
        if section.find('strong'):
            header = section.find('strong').get_text()
            content = section.find_next_sibling()
        else:
            header = ''
            content = ''
    else:
        full_text = wrap_bullets(section.get_text(separator = '\n'))
        print(type(full_text))
    #for br in section.find_all('br'):
    #    br.replace_with("\n")
    
    #full_text = section.get_text(separator = '\n')
    if not full_text:
        continue
    
    #print(full_text)
    #header_text,details = extract_bullets_from_text(full_text,header,content)
    
    #if header_text and details:
    #    headers.append(header_text)
    #    data[header_text] = details
    #else:
    details = []
    print('HELPPPPP')
    if type(full_text) == list:
        print('list')
    else:
        print('one tag')
        
    header_text = full_text.get_text()
    next_tag = full_text.find_next_sibling()
    
    if (next_tag and next_tag.name in ['ul', 'ol']):
        headers.append(header_text)
        items = next_tag.find_all('li')
        details = [item.get_text(strip=True) for item in items]
    
        data[header_text] = details 
            

# Convert to single-row DataFrame
df = pd.DataFrame({k: pd.Series(v) for k, v in data.items()})
print(df)


def extract_bullets_from_text(text,strong_header,strong_content):
    """Extract header and bullets from text"""
    print(text)
    print('\n\n\n')
    try:
        if strong_content and strong_header:
            header = strong_header
            content = strong_content.get_text().strip()
            print(header)
        else:
            if ':' in text:
                sections = text.split(':')
                #print(sections)
                header = sections[0].strip()
                content = sections[1:]
                #if sections[1].strip() != '' else header
                #print(content)
            elif '\n' in text:
                sections = text.split('\n', 1)
                #print(sections)
                #print('\n\n')
                header = sections[0].strip()
                content = sections[1].strip() if sections[1].strip() != '' else header
                
                
        if header and content:
            bullets = re.split(r'[-•·*]\s*', content)
            
            if len(bullets) <= 1:
                bullets = re.split(r'-\s+', content)
            
            bullets = [b.strip() for b in bullets if b.strip()]
            
            if bullets:
                return (header, bullets)
        
    except:
        print('Error')
        pass
    
    return (None, None)

AttributeError: 'str' object has no attribute 'get_text'